# Example: Single spin: Bayesian optimization of a gate

This is similar to the OC_Single_qubit_gate example, except that it uses Bayesian optimisation instead of gradient descent.

In [ ]:
import matplotlib.pyplot as plt

import numpy as np

from cthree.Logger import Logger
from cthree.OptimisationMap import OptimisationMap
from cthree.Quantity import Quantity
from cthree.measurement.UnitaryFidelity import UnitaryFidelity
from cthree.model.GeneratorDrive import GeneratorDrive
from cthree.model.Qubit import Qubit
from cthree.optimisers.BayesianOptimiser import BayesianOptimiser
from cthree.propagation.ScipyExpmGOAT import ScipyExpmGOAT

from cthree.model.ClosedModel import ClosedModel

from cthree.signal.SimpleGenerator import CosGenerator
from cthree.signal.Envelopes import ConstantEnvelope

For signal generation, we define a simple cosine shaped tone generator $A \cos(\omega t)$

In [ ]:
t_final = 3e-9
tone = ConstantEnvelope()
tone.t_final.set_value(t_final)
gen = CosGenerator(envelopes=[tone])

We can inspect the pre-defined parameters with

In [ ]:
params = gen.get_parameters()

in this case amplitude $A$ and frequency $\omega$.

Next, we setup the qubit system we want to control. We set the qubit frequency $\omega_q$ to be 4.8 GHz and define the Hamiltonian as $$H(t)=H_\text{drift}+H_c(t)= \frac{\omega_q}{2} \sigma_z + \Omega(t)\sigma_x$$, where $\Omega(t)$ will be supplied by the generator.

In [ ]:
FREQ = 4.8e9 * 2 * np.pi

drive = GeneratorDrive(gen, isLongitudinal=False)
controlled_qubit = Qubit(frequency=Quantity(FREQ, 0.8 * FREQ, 1.2 * FREQ), drives=[drive])
model = ClosedModel(controlled_qubit)

Textbook values for implementing an $X$ rotation on this system at a time $T$ would be $\omega=\omega_q$ and $A=\pi/T$. We use some offset from these values as initial guess to demonstrate the optimization procedure.

In [ ]:
params[0].set_value(0.5 * np.pi / t_final)
params[2].set_value(1.01 * FREQ)

In [ ]:
params

We select a propagation method, piecewise constant exponentation, and configure $\sigma_x$ as a target gate. Also we initialize the full basis at time 0 with $\mathcal{I}_2$

In [ ]:
prop = ScipyExpmGOAT(model, res=500e9)

xGate = np.array([[0.0, 1], [1, 0.0]])

prop.set_initial_state(np.identity(2))
gateFid = UnitaryFidelity(
    propagation=prop,
    gate=xGate,
    times=np.array([0.0, t_final]),
)

In [ ]:
def plotTimeEvolution():
    """Plot the time evolution of the system."""
    ts = np.linspace(0, t_final, 1001)
    states = prop.propagate(ts)
    sig = gen.generate_signal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, sig)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, np.abs(states)[:, :, 0] ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    return fig, ax


plotTimeEvolution()

As expected, we get a partial transfer and a low fidelity.

In [ ]:
gateFid.measure()

We define an optimizer and link our fidelity measure as a goal function and the parameters of the cosine tone. We also use a custom logger class to collect all samples that the optimiser takes

In [ ]:
samples = []


class CustomLogger(Logger):
    """Custom logger class definition."""

    def log(self, params: list[Quantity], infid: np.ndarray):
        """Log the list of quantities and the fidelity.

        Parameters
        ----------
        params : list[cthree.Quantity]
            List of parameters of the system.
        infid : numpy.ndarray
            Inverse of fidelity.

        """
        samples.append((params[0].get_value(), params[1].get_value()))


optmap = OptimisationMap()
optmap.add(gen, [params[0], params[2], params[3]])
opt = BayesianOptimiser(gateFid, optimisables=optmap, initialSamples=10, iterations=100)
opt.set_logger(CustomLogger())

In [ ]:
opt.optimise()

The plot shows the all the samples that the optimisation took in the two-dimensional parameter space. The red dot marks the best value.

In [ ]:
plt.figure()
plt.scatter([s[0] for s in samples[:-1]], [s[1] for s in samples[:-1]], c="blue")
plt.scatter([params[0].get_value()], [params[2].get_value()], c="red", marker="o", s=100)
plt.xlim(params[0].get_min_value(), params[0].get_max_value())
plt.ylim(params[2].get_min_value(), params[2].get_max_value())
plt.xlabel(params[0].get_name())
plt.ylabel(params[2].get_name())
plt.show()

In [ ]:
plotTimeEvolution()

We can see from the plot and optimizer output that we have found good controls.

In [ ]:
gateFid.measure()